# PDF → Chunking → Búsqueda semántica en Grafito

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpmanson/GrafitoDB/blob/main/examples/semantic/pdf_chunking_colab.ipynb)

Notebook pedagógico para **ver** cada paso del pipeline de documentos largos:

1. **PDF → texto**
2. **Chunking** (pasajes + árbol de secciones)
3. **Embeddings + búsqueda** vectorial / híbrida
4. **Expand + pack** de contexto para un LLM
5. **Visualización del grafo** en cada etapa (PyVis)

> Grafito mantiene **1 nodo = 1 vector**. Los documentos largos se modelan como muchos nodos `Chunk` / `Section` enlazados, no como multi-vector por nodo.


## 0. Instalación (Colab)

Ejecutá esta celda una vez. Luego **Runtime → Restart session** solo si Colab lo pide.


In [ ]:
# Instalación para Colab / entorno limpio
# - grafitodb[viz]: grafo + PyVis
# - pypdf / fpdf2: leer y generar PDF de ejemplo
# - sentence-transformers: embeddings reales (mejor calidad de búsqueda)

%pip install -q "grafitodb[viz]>=0.4.0" pypdf fpdf2 sentence-transformers networkx
print("OK — paquetes instalados")


## 1. Utilidades de visualización y helpers

Funciones reutilizables para mostrar el grafo en el notebook y etiquetar nodos de documento.


In [ ]:
from __future__ import annotations

import html as html_lib
import re
import tempfile
from pathlib import Path
from typing import Any, Iterable

from IPython.display import HTML, Markdown, display
from grafito import GrafitoDatabase
from grafito.document import DocumentIngestor, MarkdownChunker, TitleContextEnricher
from grafito.integrations import save_pyvis_html

# Colores por label (estudiantes: Document / Version / Section / Chunk)
DOC_COLORS = {
    "Document": "#264653",
    "DocumentVersion": "#2a9d8f",
    "Section": "#e9c46a",
    "Chunk": "#f4a261",
}


def node_display_label(node_id: Any, attrs: dict) -> str:
    props = attrs.get("properties") or {}
    labels = attrs.get("labels") or []
    if "Document" in labels:
        return f"📄 {props.get('title') or props.get('document_key') or node_id}"
    if "DocumentVersion" in labels:
        return f"v{props.get('generation', '?')} {props.get('status', '')}"
    if "Section" in labels:
        title = props.get("title") or "?"
        return f"§ {title[:28]}"
    if "Chunk" in labels:
        text = (props.get("text") or "")[:36].replace("\n", " ")
        seq = props.get("global_seq", "?")
        return f"#{seq} {text}…"
    return str(props.get("name") or props.get("title") or node_id)


def show_graph(
    db: GrafitoDatabase,
    *,
    title: str = "",
    include_ids: set[int] | None = None,
    highlight_ids: set[int] | None = None,
    height: str = "520px",
    physics: str = "spread",
) -> None:
    """Renderiza un subgrafo con PyVis embebido en la celda."""
    G = db.to_networkx()
    if include_ids is not None:
        keep = set(include_ids)
        G = G.subgraph([n for n in G.nodes if n in keep]).copy()

    # Marcar highlights en properties para color
    highlight_ids = highlight_ids or set()
    for nid in list(G.nodes):
        attrs = G.nodes[nid]
        props = dict(attrs.get("properties") or {})
        labels = attrs.get("labels") or []
        if nid in highlight_ids:
            props["_viz_color"] = "#e63946"
        else:
            color = DOC_COLORS.get(labels[0] if labels else "", "#8ecae6")
            props["_viz_color"] = color
        attrs["properties"] = props

    path = Path(tempfile.gettempdir()) / "grafito_step_graph.html"
    save_pyvis_html(
        G,
        path=str(path),
        notebook=False,
        directed=True,
        color_by_label=False,
        node_color_attr="_viz_color",
        label_fn=node_display_label,
        physics=physics,
        height=height,
        width="100%",
        bgcolor="#ffffff",
        font_color="#222222",
        cdn_resources="in_line",
    )
    raw = path.read_text(encoding="utf-8")
    if title:
        display(Markdown(f"### {title}"))
    # iframe para Colab / Jupyter
    escaped = html_lib.escape(raw)
    # Prefer direct HTML embed when possible
    display(HTML(f'<div style="border:1px solid #ddd;border-radius:8px;overflow:hidden">{raw}</div>'))


def legend() -> None:
    items = "".join(
        f'<span style="display:inline-block;margin:4px 12px 4px 0">'
        f'<span style="display:inline-block;width:12px;height:12px;background:{c};'
        f'border-radius:50%;margin-right:6px"></span>{name}</span>'
        for name, c in DOC_COLORS.items()
    )
    items += (
        '<span style="display:inline-block;margin:4px 12px 4px 0">'
        '<span style="display:inline-block;width:12px;height:12px;background:#e63946;'
        'border-radius:50%;margin-right:6px"></span>Hit / expand</span>'
    )
    display(HTML(f"<div style='font-family:sans-serif;font-size:14px'>{items}</div>"))


print("Helpers listos")
legend()


## 2. Obtener un PDF

**Opción A (recomendado en clase):** subí tu PDF.  
**Opción B:** generamos un PDF de ejemplo multi-sección (runbook inventado).


In [ ]:
from pypdf import PdfReader
from fpdf import FPDF

# --- Opción A: descomentar en Colab para subir un PDF ---
# from google.colab import files
# uploaded = files.upload()
# PDF_PATH = next(iter(uploaded))

PDF_PATH = "sample_runbook.pdf"


def write_sample_pdf(path: str = PDF_PATH) -> str:
    """PDF multi-sección para demostrar hierarchy + fences."""
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", size=14)

    sections = [
        ("Triaging slow graph queries",
         "When query latency spikes, work through these steps before scaling hardware."),
        ("Check connection pool",
         "Connection pool exhaustion often looks like random timeouts under load. "
         "Symptoms include intermittent timeout waiting for connection and rising queue depth."),
        ("Inspect Cypher plans",
         "Look for cartesian products and unbounded variable-length paths. "
         "Prefer bounded patterns like KNOWS*1..3 and always set a LIMIT for exploratory queries."),
        ("Escalate with evidence",
         "If pool and plans look fine, capture a profile and open an issue with document_key, "
         "sample Cypher, and wall-clock timings for the on-call engineer."),
    ]
    for title, body in sections:
        pdf.set_font("Helvetica", "B", 16)
        pdf.multi_cell(0, 10, title)
        pdf.ln(2)
        pdf.set_font("Helvetica", size=12)
        pdf.multi_cell(0, 7, body)
        pdf.ln(6)
    # Note: PDF text extraction is plain text; hierarchy uses ATX headings we inject after extract
    pdf.output(path)
    return path


# Si no hay upload, crear sample
if not Path(PDF_PATH).exists():
    write_sample_pdf(PDF_PATH)
    print(f"PDF de ejemplo creado: {PDF_PATH}")
else:
    print(f"Usando PDF existente: {PDF_PATH}")


## 3. Extraer texto del PDF

`pypdf` devuelve texto plano (sin markdown). Para que el chunker jerárquico vea secciones, convertimos títulos conocidos o usamos heurística de líneas en mayúsculas / cortas como headings.

En un documento real con buen extract (o OCR), podés mapear headings a `#` / `##` con tu propio parser.


In [ ]:
def pdf_to_text(path: str) -> str:
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        t = page.extract_text() or ""
        pages.append(t)
        print(f"Página {i+1}: {len(t)} caracteres")
    return "\n\n".join(pages)


def promote_headings(plain: str, known_titles: list[str] | None = None) -> str:
    """Convierte títulos conocidos en ATX para MarkdownChunker."""
    text = plain
    if known_titles:
        for i, title in enumerate(known_titles):
            level = "#" if i == 0 else "##"
            # Reemplazo de línea que contiene el título
            pattern = re.compile(rf"^{re.escape(title)}\s*$", re.MULTILINE)
            text = pattern.sub(f"{level} {title}", text)
    return text


RAW = pdf_to_text(PDF_PATH)
KNOWN = [
    "Triaging slow graph queries",
    "Check connection pool",
    "Inspect Cypher plans",
    "Escalate with evidence",
]
DOC_TEXT = promote_headings(RAW, KNOWN)

print("\n--- Vista previa (primeros 800 caracteres) ---")
print(DOC_TEXT[:800])
print("\n... total:", len(DOC_TEXT), "caracteres")


## 4. Crear base, embeddings e **ingestar** el documento

Usamos `DocumentIngestor` + `MarkdownChunker`:

- Nodo `Document` (raíz)
- `DocumentVersion` (generación ACTIVE)
- `Section` (árbol ToC)
- `Chunk` / passages (texto indexable + 1 vector c/u)


In [ ]:
from sentence_transformers import SentenceTransformer
from grafito.embedding_functions.base import EmbeddingFunction


class STEmbedder(EmbeddingFunction):
    """Adapter mínimo SentenceTransformer → API de Grafito."""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self._dim = int(self.model.get_sentence_embedding_dimension())

    def __call__(self, input: list[str]) -> list[list[float]]:
        return self.model.encode(input, normalize_embeddings=True).tolist()

    @staticmethod
    def name() -> str:
        return "sentence_transformer_notebook"

    def default_space(self) -> str:
        return "cosine"

    def supported_spaces(self) -> list[str]:
        return ["cosine"]

    @staticmethod
    def build_from_config(config: dict) -> "STEmbedder":
        return STEmbedder(config.get("model_name", "sentence-transformers/all-MiniLM-L6-v2"))

    def get_config(self) -> dict:
        return {"model_name": self.model_name, "dim": self._dim}

    @staticmethod
    def validate_config(config: dict) -> None:
        return None

    @property
    def dimension(self) -> int:
        return self._dim


print("Cargando modelo de embeddings (primera vez puede tardar)…")
embedder = STEmbedder()
print("dim =", embedder.dimension)

db = GrafitoDatabase(":memory:")
db.create_vector_index(
    "docs_chunks",
    dim=embedder.dimension,
    backend="bruteforce",
    embedding_function=embedder,
    options={"metric": "cosine"},
)

ing = DocumentIngestor(
    db,
    chunker=MarkdownChunker(max_chars=500, overlap=50),
    embed_index="docs_chunks",
    configure_fts=db.has_fts5(),
    enricher=TitleContextEnricher(),
    hierarchy="auto",
)

result = ing.ingest(
    DOC_TEXT,
    document_key="class/pdf-demo",
    title="Triaging slow graph queries",
    source=PDF_PATH,
    embed=True,
)

print(result)
print(f"hierarchy={result.hierarchy} sections={result.n_sections} passages={result.n_passages}")


### 4.1 Visualización post-ingest

Leyenda de colores + grafo completo del documento (Document → Version → Sections → Chunks).


In [ ]:
legend()

# Todos los nodos del documento gestionado
managed = db.match_nodes(properties={"managed_by": "grafito.document"})
doc_nodes = db.match_nodes(labels=["Document"], properties={"document_key": "class/pdf-demo"})
ids = {n.id for n in managed} | {n.id for n in doc_nodes}

show_graph(
    db,
    title="Grafo tras el ingest (estructura completa)",
    include_ids=ids,
    physics="spread",
    height="560px",
)

# Tabla de contenidos
print("\nToC:")
for sec in ing.toc("class/pdf-demo"):
    print(f"  [{sec.node_key}] L{sec.level} {sec.title}  (chunks directos≈{(sec.metadata or {}).get('n_chunks')})")
    for c in sec.children:
        print(f"      [{c.node_key}] L{c.level} {c.title}")


### 4.2 Solo el árbol de secciones (sin passages)

Útil para validar que el PDF se segmentó en headings correctos.


In [ ]:
parent = doc_nodes[0]
sec_nodes = [
    n for n in db.match_nodes(labels=["Section"])
    if n.properties.get("owner_document_id") == parent.id
]
ver_nodes = [
    n for n in db.match_nodes(labels=["DocumentVersion"])
    if n.properties.get("owner_document_id") == parent.id
]
tree_ids = {parent.id} | {n.id for n in sec_nodes} | {n.id for n in ver_nodes}

show_graph(
    db,
    title="Solo Document + Version + Sections (ToC)",
    include_ids=tree_ids,
    physics="spread",
    height="480px",
)

## 5. Búsqueda semántica

Buscamos por significado. Luego resaltamos el hit en el grafo.


In [ ]:
QUERY = "connection pool timeout under load"
hits = ing.search(QUERY, k=4)

print(f"Query: {QUERY!r}\n")
for i, h in enumerate(hits, 1):
    text = (h.node.properties.get("text") or "").replace("\n", " ")
    print(f"{i}. score={h.score:.3f}  seq={h.global_seq}  section_id={h.node.properties.get('section_node_id')}")
    print(f"   {text[:140]}…\n")

hit_ids = {h.node.id for h in hits}
# incluir ancestros de sección del top hit para contexto visual
extra = set()
if hits:
    ex0 = ing.expand(hits[0].node, window=0, include_ancestors=True)
    if ex0.section:
        extra.add(ex0.section.id)
    extra.update(a.id for a in ex0.ancestors)
    extra.add(parent.id)

show_graph(
    db,
    title="Hits de búsqueda (rojo) + ancestros del top hit",
    include_ids=ids,
    highlight_ids=hit_ids | extra,
    physics="spread",
)


## 6. Expand + pack (contexto para un LLM)

A partir del mejor hit, tomamos vecinos en orden de lectura (`global_seq ± window`) y empaquetamos texto con citas.


In [ ]:
if not hits:
    raise RuntimeError("No hay hits — revisá el texto extraído del PDF o la query.")

top = hits[0]
expanded = ing.expand(top.node, window=1, include_parent=True, include_ancestors=True)
packed = ing.pack(expanded, max_chars=1800, include_citations=True)

print("Section:", None if not expanded.section else expanded.section.properties.get("title"))
print("Ancestors:", [a.properties.get("title") for a in expanded.ancestors])
print("Passages in window:", [p.properties.get("global_seq") for p in expanded.passages])
print("truncated:", packed.truncated)
print("\n----- PACKED CONTEXT -----\n")
print(packed.text)

window_ids = {p.id for p in expanded.passages}
if expanded.section:
    window_ids.add(expanded.section.id)
window_ids.update(a.id for a in expanded.ancestors)
window_ids.add(parent.id)

show_graph(
    db,
    title="Ventana de expand (passages vecinos + sección + documento)",
    include_ids=window_ids,
    highlight_ids={top.node.id},
    physics="compact",
    height="480px",
)


## 7. Búsqueda híbrida (vector + FTS + RRF)

Si SQLite tiene FTS5, combinamos keyword y semántica. Los scores RRF **no** son comparables a cosine crudo.


In [ ]:
if db.has_fts5():
    hyb = ing.hybrid_search("pool timeout", k=4)
    print("Hybrid RRF results:\n")
    for i, h in enumerate(hyb, 1):
        text = (h.node.properties.get("text") or "").replace("\n", " ")[:100]
        print(f"{i}. rrf={h.score:.4f}  {text}…")
    show_graph(
        db,
        title="Hits hybrid_search (rojo)",
        include_ids=ids,
        highlight_ids={h.node.id for h in hyb},
    )
else:
    print("FTS5 no disponible en este SQLite — se omite hybrid_search.")


## 8. Ejercicios para estudiantes

1. **Cambiá la query** a algo sobre *Cypher plans* y observá qué passage se ilumina.
2. Subí **tu propio PDF** y ajustá `promote_headings` (o generá markdown a mano).
3. Probá `window=0` vs `window=2` en `expand` y compará el pack.
4. Desactivá hierarchy: `hierarchy=False` en `DocumentIngestor` y compará el grafo (solo Chunks planos).
5. (Avanzado) Usá `tree_select` con un LLM que devuelva `node_key` del ToC.

### Referencias

- Docs: [Document Chunking](https://jpmanson.github.io/GrafitoDB/search/document-chunking/)
- Ejemplo offline: `examples/semantic/document_chunking.py`
- Visualización: [Visualization](https://jpmanson.github.io/GrafitoDB/integrations/visualization/)


In [ ]:
# Limpieza opcional
db.close()
print("Sesión cerrada. ¡Listo para experimentar!")
